<a href="https://colab.research.google.com/github/henriquecrispim/global-market-sentiment-pipeline/blob/main/global_market_sentiment_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PIPELINE DE INGESTÃO MASSIVA E ANÁLISE DE SENTIMENTO MULTIREGIONAL
# ==============================================================================

# 1. Instalação das dependências reais de produção
!pip install pandas nltk plotly feedparser --quiet

import feedparser
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import time
from datetime import datetime

# Garante o funcionamento dinâmico dos gráficos no ecossistema Colab
pio.renderers.default = 'colab'

# Inicializa o cérebro de Processamento de Linguagem Natural (PLN) focado em finanças
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

print("⚡ [ENGINE-INIT] Conectando à malha de Big Data... Varrendo jornais mundiais.")

# ==============================================================================
# 2. MALHA DE CAPTURA (BIG DATA LAYER) - 19 FONTES REAIS EM PRODUÇÃO
# ==============================================================================
FONTES_JORNAIS = {
    # --- JORNAIS INTERNACIONAIS (Visão Macro, Câmbio e Mercados Globais) ---
    'Yahoo Finance': 'https://finance.yahoo.com/news/rssindex',
    'Reuters International': 'https://services.reuters.com/reuters/USAndInternationalFinancialNews',
    'CNBC Finance': 'https://search.cnbc.com/rs/search/combinedseo.xml?partnerId=401&id=15839069',
    'MarketWatch Top Stories': 'https://feeds.brg.blogs.com/marketwatch/topstories/',
    'CNN Business': 'http://rss.cnn.com/rss/money_latest.rss',
    'WSJ Business': 'https://feeds.a.dj.com/rss/WSJcomUSBusiness.mid',
    'BBC Business News': 'http://feeds.bbci.co.uk/news/business/rss.xml',
    'Fortune Magazine': 'https://fortune.com/feed/fortune-feeds',
    'Forbes Financials': 'https://www.forbes.com/business/feed/',
    'The Economist Markets': 'https://www.economist.com/finance-and-economics/rss.xml',

    # --- JORNAIS NACIONAIS (Cenário Fiscal, Juros Copom, B3 e Boletim Focus) ---
    'G1 Economia': 'https://g1.globo.com/dynamo/economia/rss2.xml',
    'Valor Econômico': 'https://valor.globo.com/rss/economia/',
    'InfoMoney Mercados': 'https://www.infomoney.com.br/mercados/feed/',
    'CNN Brasil Business': 'https://www.cnnbrasil.com.br/business/feed/',
    'Estadão Economia': 'https://www.estadao.com.br/economia/rss/',
    'Exame Negócios': 'https://exame.com/economia/feed/',
    'Investing.com Brasil': 'https://br.investing.com/rss/news_285.rss',
    'Money Times': 'https://www.moneytimes.com.br/feed/',
    'BBC Brasil Economia': 'https://www.bbc.com/portuguese/topics/c7zp57yyz21t/rss.xml'
}

noticias_pipeline = []

# Varre todas as fontes assincronamente extraindo as últimas 25 manchetes de cada uma
for jornal, url_feed in FONTES_JORNAIS.items():
    try:
        print(f"📡 [SCRAPER] Conectando ao servidor: {jornal}...")
        parsed_feed = feedparser.parse(url_feed)

        # Coleta até 25 manchetes por portal
        for entry in parsed_feed.entries[:25]:
            # Classificação geográfica automática com base no portal
            regiao = 'Nacional (Brasil)' if jornal in [
                'G1 Economia', 'Valor Econômico', 'InfoMoney Mercados', 'CNN Brasil Business',
                'Estadão Economia', 'Exame Negócios', 'Investing.com Brasil', 'Money Times', 'BBC Brasil Economia'
            ] else 'Internacional (Global)'

            noticias_pipeline.append({
                'Portal': jornal,
                'Geografia': regiao,
                'Manchete': entry.title,
                'Link': getattr(entry, 'link', ''),
                'Data_Captura': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
    except Exception as e:
        print(f"⚠️ [AVISO] Falha de handshake com o portal {jornal}. Pulando fonte...")

# Estruturação e normalização dos dados não-estruturados com Pandas
df_corporativo = pd.DataFrame(noticias_pipeline)

if df_corporativo.empty:
    print("❌ [ERRO] Falha crítica na camada de ingestão. Nenhuma manchete coletada.")
else:
    print(f"\n🚀 [DATA-SUCCESS] Ingestão Concluída! {len(df_corporativo)} manchetes extraídas da internet agora.")

    # ==============================================================================
    # 3. CAMADA DE INTELIGÊNCIA ARTIFICIAL E PROCESSAMENTO LÉXICO
    # ==============================================================================
    print("🤖 [PLN-PROCESSING] Calculando métricas de polaridade textual (VADER Engine)...")

    # Aplica o cálculo matemático do Compound Score (-1 a +1)
    df_corporativo['Sentiment_Score'] = df_corporativo['Manchete'].apply(lambda x: sia.polarity_scores(x)['compound'])

    # Define as zonas de corte de volatilidade do humor
    df_corporativo['Classificacao'] = pd.cut(
        df_corporativo['Sentiment_Score'],
        bins=[-1.0, -0.05, 0.05, 1.0],
        labels=['Pessimista (Bearish)', 'Neutro / Estável', 'Otimista (Bullish)']
    )

    # Consolidação estatística cruzada por Região e Estado de Humor
    df_analytics = df_corporativo.groupby(['Geografia', 'Classificacao'], observed=False).size().reset_index(name='Volume')

    # Cálculo do Índice de Sentimento Médio Ponderado por Mercado
    score_medio_br = df_corporativo[df_corporativo['Geografia'] == 'Nacional (Brasil)']['Sentiment_Score'].mean()
    score_medio_gl = df_corporativo[df_corporativo['Geografia'] == 'Internacional (Global)']['Sentiment_Score'].mean()

    print("\n" + "="*75)
    print(f"📊 ÍNDICE DE SENTIMENTO MACRO NACIONAL (Brasil): {score_medio_br:+.4f}")
    print(f"📊 ÍNDICE DE SENTIMENTO MACRO INTERNACIONAL (Global): {score_medio_gl:+.4f}")
    print("="*75 + "\n")

    # ==============================================================================
    # 4. ENGENHARIA GRÁFICA MULTIREGIONAL COMPATÍVEL (DASHBOARD)
    # ==============================================================================
    # Gera um gráfico de barras comparando lado a lado o humor nacional vs internacional
    fig = px.bar(
        df_analytics,
        x='Classificacao',
        y='Volume',
        color='Classificacao',
        facet_col='Geografia',
        color_discrete_map={
            'Otimista (Bullish)': '#27ae60',   # Verde Clássico de Mercado
            'Pessimista (Bearish)': '#c0392b',  # Vermelho de Alerta/Aversão
            'Neutro / Estável': '#7f8c8d'      # Cinza Neutro
        },
        title=f"Dashboard Comparativo de Sentimento da Mídia — Painel Multiregional Live"
    )

    fig.update_layout(
        xaxis_title="Humor Computado via PLN",
        yaxis_title="Volume de Notícias Reais Ingeridas",
        font=dict(family="Arial", size=11),
        margin=dict(r=20, t=70, l=20, b=20)
    )

    # Salva o arquivo HTML estático para o portfólio
    fig.write_html("dashboard_sentimento_massivo.html")

    # Renderiza o painel na tela
    fig.show()

    # Exibe o Relatório Auditável contendo as notícias de verdade que o código leu
    print("\n📋 BASE DE DADOS INTEGRADA (AMOSTRA DE REGISTROS REAL-TIME):")
    pd.set_option('display.max_colwidth', None)
    display(df_corporativo[['Portal', 'Geografia', 'Classificacao', 'Sentiment_Score', 'Manchete']].head(15))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 798.5 kB/s eta 0:00:00
⚡ [ENGINE-INIT] Conectando à malha de Big Data... Varrendo jornais mundiais.
📡 [SCRAPER] Conectando ao servidor: Yahoo Finance...
📡 [SCRAPER] Conectando ao servidor: Reuters International...
📡 [SCRAPER] Conectando ao servidor: CNBC Finance...
📡 [SCRAPER] Conectando ao servidor: MarketWatch Top Stories...
📡 [SCRAPER] Conectando ao servidor: CNN Business...
📡 [SCRAPER] Conectando ao servidor: WSJ Business...
📡 [SCRAPER] Conectando ao servidor: BBC Business News...
📡 [SCRAPER] Conectando ao servidor: Fortune Magazine...
📡 [SCRAPER] Conectando ao servidor: Forbes Financials...
📡 [SCRAPER] Conectando ao servidor: The Economist Markets...
📡 [SCRAPER] Conectando ao servidor: G1 Economia...
📡 [SCRAPER] Conectando ao servidor: Valor Econômico...
📡 [SCRAPER] Conectando ao servidor: InfoMoney Mercados...
📡 [SCRAPER] Conectando ao servidor: CNN Brasil Business...
📡 [SCRAPER] Con


📋 BASE DE DADOS INTEGRADA (AMOSTRA DE REGISTROS REAL-TIME):


,Portal,Geografia,Classificacao,Sentiment_Score,Manchete
0,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.3818,Best Buy and Apple flag a price shock for shoppers
1,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.2263,"British American Tobacco Is Cutting 5,500 Jobs. How to Play the High-Yield Dividend Stock Here."
2,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.3818,"Dear Taiwan Semi Stock Fans, Mark Your Calendars for July 16"
3,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.4927,Dell Stock Has Very Attractive Short-Put Yields - Over 3.8% for the Next 2 Weeks
4,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.8020,"Got $2M saved for retirement? Get ready for everything to change, and not always for the better. Dodge 5 money traps now"
5,Yahoo Finance,Internacional (Global),Neutro / Estável,0.0000,"If You'd Parked $1,000 in the iShares Semiconductor ETF 5 Years Ago, Here's What You'd Have Today"
6,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.4215,How Honeywell Technologies’ (HON) Post-Spin Structure Creates a Cleaner Automation Thesis
7,Yahoo Finance,Internacional (Global),Pessimista (Bearish),-0.1280,How NVIDIA Corporation’s (NVDA) AI Factory Model Could Turn Blackwell Demand Into Usage-Linked Revenue
8,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.4767,"How Advanced Micro Devices, Inc.’s (AMD) Server CPU Demand Is Broadening Its AI Growth Story"
9,Yahoo Finance,Internacional (Global),Otimista (Bullish),0.0516,"Why Micron Technology, Inc.’s (MU) Memory Tightness Is Turning AI Demand Into Cleaner Earnings Momentum"
